# Question 1 -- Report figures & tables

**Purpose of this notebook.** Assemble the report-ready figures, tables and commentary for Question 1 (price-elastic consumption, PV, tariffs), from the CSV / TXT files that `main.py` already writes under `results/Q1_caseA/` (one row per hour, plus a small summary of the objective and scalar duals).

**This notebook does not reimplement the model.** All solving happens in `src/model.py` (subclassed per question) and is run through `main.py`. Here we only load already-computed results and turn them into the plots/tables/discussion that go into the report.

**Workflow**
1. Complete the model for this question in `src/` and run it, e.g. `python main.py --question Q1_caseA --scenarios`.
2. Re-run the cells below -- they read the CSVs written under `results/`.
3. Fill in the markdown commentary cells (marked `_TODO_`) with your actual analysis for the report.
4. Before committing: `Kernel -> Restart & Run All` once, to check the notebook still runs top to bottom against the latest `results/`, then consider clearing outputs (`Edit -> Clear Outputs`) so git diffs stay small.

Section headers below follow the sub-question lettering of the assignment instructions (`course_materials/Assignment_1_Instructions.html`, grading table), so everyone on the team knows where to put what.


## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Make `src` importable regardless of where Jupyter was launched from.
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.plotting import plot_inputs, plot_schedule, plot_duals, plot_scenario_comparison

RESULTS_DIR = ROOT_DIR / "results"
%matplotlib inline


In [2]:
def load_hourly(question: str, tag: str = "") -> pd.DataFrame:
    """Read results/<question>/<question>[_<tag>]_hourly.csv (written by Results.save())."""
    stem = f"{question}{'_' + tag if tag else ''}"
    path = RESULTS_DIR / question / f"{stem}_hourly.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found yet -- run `python main.py --question {question}"
            f"{' --scenarios' if tag else ''}` from the repo root first."
        )
    return pd.read_csv(path, index_col="hour")


def load_summary(question: str, tag: str = "") -> str:
    """Read results/<question>/<question>[_<tag>]_summary.txt (objective + scalar duals)."""
    stem = f"{question}{'_' + tag if tag else ''}"
    path = RESULTS_DIR / question / f"{stem}_summary.txt"
    return path.read_text(encoding="utf-8") if path.exists() else f"(missing: {path})"


## Sanity check: does the setup work?

Before anything else, check that your setup matches **README.md, Section 1** ("Setup"):
the Python environment is installed, `gurobipy` can actually find a Gurobi licence, and
`python main.py` runs end to end (README 1.3).

Right now `FlexibleConsumerModel.build()` in `src/model.py` is still `TODO`, so the expected
outcome below is the model printing `[skipped] The model has no constraints ...` and stopping
there -- that is normal at this stage (README 1.3 says so explicitly), **not** a bug. What this
section actually checks for is everything *around* that: packages import, the Gurobi licence is
valid, `main.py` runs without a Python traceback, and it writes the input-data figure to
`results/`. Once `build()` is implemented, re-run this section and the `[skipped]` message and
the "no constraints" check below should both disappear.


In [3]:
import sys
import gurobipy as gp
import numpy as np
import pandas as pd
import matplotlib

print(f"Python     : {sys.version.split()[0]}")
print(f"gurobipy   : {gp.gurobi.version()}")
print(f"numpy      : {np.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"matplotlib : {matplotlib.__version__}")


Python     : 3.12.3
gurobipy   : (13, 0, 3)
numpy      : 2.5.3
pandas     : 3.0.5
matplotlib : 3.11.1


In [4]:
# README 1.2: gurobipy ships with a size-restricted licence that is enough for this assignment
# (models up to 2000 variables/constraints) -- this should succeed with no error out of the box.
try:
    _m = gp.Model("licence_check")
    _x = _m.addVar(name="x")
    _m.setObjective(_x, gp.GRB.MAXIMIZE)
    _m.addConstr(_x <= 1)
    _m.optimize()
    assert _m.Status == gp.GRB.OPTIMAL and abs(_x.X - 1) < 1e-6
    print("Gurobi licence OK - trivial model solved (x* =", _x.X, ")")
except gp.GurobiError as e:
    print("Gurobi licence problem:", e)
    print("-> see README.md Section 1.2 (request an academic licence with your DTU e-mail, "
          "then `grbgetkey <your-key>` on the DTU network/VPN).")


Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-1065G7 CPU @ 1.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1 rows, 1 columns and 1 nonzeros (Max)
Model fingerprint: 0x7f299f02
Model has 1 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 1 rows and 1 columns
Presolve time: 0.02s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.03 seconds (0.00 work units)
Optimal objective  1.000000000e+00
Gurobi licence OK - trivial model solved (x* = 1.0 )


In [5]:
import subprocess

# README 1.3: `python main.py` should run without a traceback, load Q1_caseA, print a data
# summary, and save results/Q1_caseA/inputs.png -- even before build() is implemented.
proc = subprocess.run(
    [sys.executable, str(ROOT_DIR / "main.py"), "--question", "Q1_caseA"],
    cwd=ROOT_DIR, capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
assert proc.returncode == 0, "main.py exited with an error -- see stderr above"
assert "[skipped]" in proc.stdout or "status" in proc.stdout.lower(), (
    "Unexpected output from main.py -- check it against README 1.3"
)
print("main.py ran successfully.",
      "(model not implemented yet -- '[skipped]' above is expected)" if "[skipped]" in proc.stdout
      else "(model solved end to end)")


Case                 : Q1_caseA
Consumer             : C_01_A
Energy price         : mean 1.32, min 0.85, max 2.50 DKK/kWh
Import/export tariff : 0.50 / 0.40 DKK/kWh
PV                   : 6.0 kW peak, 26.9 kWh available/day, marginal cost 1.41 DKK/kWh
Load bounds          : 0.0 - 6.0 kWh/h
Consumption utility  : 1.43
Min daily energy     : None
Reference profile    : no
Disutility lin/quad  : None / None
Battery              : no
Next-day forecasts   : yes 

Restricted license - for non-production use only - expires 2027-11-29
[skipped] The model has no constraints: complete FlexibleConsumerModel.build() in src/model.py first.

Outputs written to C:\Users\Alejandro\Desktop\46750 Optimization\Assignment_1\Assignment-1-optimization-power-markets\results\Q1_caseA

main.py ran successfully. (model not implemented yet -- '[skipped]' above is expected)


In [6]:
# main.py should have written the input-data figure even before build() is implemented.
inputs_png = RESULTS_DIR / "Q1_caseA" / "inputs.png"
assert inputs_png.exists(), f"{inputs_png} missing -- did main.py run from the repo root?"
print(f"OK: {inputs_png} exists ({inputs_png.stat().st_size} bytes)")

# Once build() is implemented these will also exist -- harmless (just informative) until then.
for extra in ["Q1_caseA_hourly.csv", "Q1_caseA_summary.txt", "schedule.png", "duals.png"]:
    p = RESULTS_DIR / "Q1_caseA" / extra
    print(f"{'[ok]  ' if p.exists() else '[--]  '}{p.name}"
          + ("" if p.exists() else "  (expected once build() is implemented)"))


OK: c:\Users\Alejandro\Desktop\46750 Optimization\Assignment_1\Assignment-1-optimization-power-markets\results\Q1_caseA\inputs.png exists (56472 bytes)
[--]  Q1_caseA_hourly.csv  (expected once build() is implemented)
[--]  Q1_caseA_summary.txt  (expected once build() is implemented)
[--]  schedule.png  (expected once build() is implemented)
[--]  duals.png  (expected once build() is implemented)


## (a) -- Formulation, properties, hourly decomposition and its meaning

_TODO: no figures/tables expected here -- this sub-question is answered as text/derivation in the report. Optional: keep your working/notes for it here so the team can see it while drafting._

## (b) -- Lagrange function, dual problem, KKT conditions, strong duality, necessity/sufficiency

_TODO: no figures/tables expected here -- this sub-question is answered as text/derivation in the report. Optional: keep your working/notes for it here so the team can see it while drafting._

## (c) -- Interpretation of the duals (balance, PV limit, load bounds) and their uniqueness

_TODO: no figures/tables expected here -- this sub-question is answered as text/derivation in the report. Optional: keep your working/notes for it here so the team can see it while drafting._

## (d) -- Structure of the optimal solutions from the KKT conditions (price ladder), incl. (d).i-v

_TODO: no figures/tables expected here -- this sub-question is answered as text/derivation in the report. Optional: keep your working/notes for it here so the team can see it while drafting._

## (e) -- Implementation, including extraction of the dual variables (the code itself, not a report figure)

**Working area for 1.(e).** The model itself lives in `FlexibleConsumerModel.build()` in `src/model.py`; the cells below only import it, run it and check the result. Edit `src/model.py`, then re-run the cells from top to bottom (autoreload picks up the changes). Once everything passes, confirm with `python main.py --question Q1_caseA` from the terminal.

In [ ]:
# Reload src/*.py automatically every time it is edited (no kernel restart needed).
%load_ext autoreload
%autoreload 2


In [ ]:
# --- 1. Load the data of the case ------------------------------------------------
from src.data_loader import load_question
from src.model import FlexibleConsumerModel

data = load_question("Q1_caseA")
print(data.summary())


In [ ]:
# --- 2. Build and solve ---------------------------------------------------------
# Fails with "The model has no constraints" until build() in src/model.py is written.
model = FlexibleConsumerModel(data).build()
results = model.solve()
print(results)


In [ ]:
# --- 3. Primal checks -----------------------------------------------------------
hr = results.hourly
p_imp = data.energy_price + data.import_tariff     # effective import price, DKK/kWh
p_exp = data.energy_price - data.export_tariff     # effective export price, DKK/kWh
display(hr.round(3))

# TODO (a): power balance. Print the maximum absolute value of  load - (pv + import - export).
# TODO (b): bounds. Check that Lmin <= load <= Lmax and 0 <= pv <= pv_available in every hour.
# TODO (c): no simultaneous import and export. Count the hours with both > 1e-6 (expect 0).
# TODO (d): objective. Compute utility = sum(uL * load) and
#           procurement cost = sum(p_imp*import - p_exp*export + cPV*pv);
#           check that utility - cost equals results.objective.


In [ ]:
# --- 4. Dual checks -------------------------------------------------------------
# The names of the dual columns are "dual_<name>", where <name> is the key you used in
# self.con[...] inside build(). Print them first and adapt the names below.
print([c for c in hr.columns if c.startswith("dual_")])

# TODO (a): in the hours where the consumer imports (import > 1e-6), the dual of the balance
#           should equal p_imp. Print max |dual - p_imp| over those hours.
# TODO (b): same for export hours: the dual of the balance should equal p_exp.
# TODO (c): check the four stationarity conditions of your 1.(b), one at a time, e.g. for the load:
#           -uL + lambda - mu_L_lo + mu_L_up == 0   (each with your own sign convention).
#           Print the maximum absolute error of each.
# TODO (d): print the sign of every dual (all mu should be >= 0). If not, revisit the sign of the
#           inequalities in build() (write them as  expression <= 0).


## (f) -- Numerical validation: base cases A and B, hour-by-hour ladder verification, comparison, duals

In [7]:
# Load the run(s) needed for (f). Adjust the tags to match the scenarios you actually saved.
tags = ["base"]
runs = {tag: load_hourly("Q1_caseA", tag if tag != "base" else "") for tag in tags}
for tag in tags:
    print(f"--- {tag} ---")
    print(load_summary("Q1_caseA", tag if tag != "base" else ""))


FileNotFoundError: c:\Users\Alejandro\Desktop\46750 Optimization\Assignment_1\Assignment-1-optimization-power-markets\results\Q1_caseA\Q1_caseA_hourly.csv not found yet -- run `python main.py --question Q1_caseA` from the repo root first.

In [ ]:
# TODO: report figure(s) for (f).
# e.g. fig, ax = plt.subplots(figsize=(9, 4))
#      ax.step(runs["base"].index, runs["base"]["load"], where="mid", label="load")
#      ...
#      fig.savefig(ROOT_DIR / "notebooks" / "figures" / "Q1_caseA_(f).png", dpi=200)


_TODO: discussion / commentary for the report goes here._